# Week 8: Heap & Priority Queue — Always Know the Most Important Item
## PHASE 4: Hierarchical Structures

*📚 Data Structures & Algorithms · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Understand what a priority queue is and identify real-world examples
2. Explain the heap data structure: a complete binary tree with the heap property
3. Use Python's `heapq` module for efficient priority queue operations
4. Find the smallest or largest k elements efficiently
5. Benchmark `sorted()` vs `heapq.nsmallest()` and interpret the results
6. Recognise when to use a heap vs sorting

## 🎯 Core Mastery Connection

When you always need the min/max element, a heap gives O(log n) insert + O(1) peek — far better than re-sorting every time. This week you will benchmark `sorted()` vs `heapq` to see empirically when and why heaps win. Remember: predict first, then measure. Does reality match your prediction?

---
## 📦 Setup

Run this cell first to load the required packages.

In [ ]:
import matplotlib.pyplot as plt

import heapq
import random
import time

---
## Part 1: What Is a Priority Queue?

A **priority queue** is like a regular queue (first-in, first-out) but with a twist: **the most important item comes out first**, regardless of when it was added.

### Real-World Analogies

| Scenario | Priority Rule | Regular Queue? |
|----------|--------------|----------------|
| **Emergency Room** | Most critical patient first | No — a broken finger waits for a heart attack |
| **Print Queue** | Urgent documents before large reports | No — priority matters |
| **Airport Boarding** | First class, then business, then economy | No — ticket class matters |
| **Operating System** | High-priority processes run first | No — system tasks beat user tasks |

### Key Operations

| Operation | Description | Time Complexity |
|-----------|------------|----------------|
| **Insert** | Add an item with a priority | O(log n) |
| **Extract min/max** | Remove the highest-priority item | O(log n) |
| **Peek** | Look at the highest-priority item (without removing) | O(1) |

**Figure 1.1** — A simple priority queue using a sorted list (inefficient but intuitive)

In [ ]:
# Naive approach: use a sorted list
# Lower number = higher priority

class NaivePriorityQueue:
    def __init__(self):
        self.items = []

    def insert(self, priority, item):
        self.items.append((priority, item))
        self.items.sort()  # O(n log n) every time!

    def extract_min(self):
        return self.items.pop(0)  # O(n) due to shifting

    def peek(self):
        return self.items[0]

# ER triage simulation
er = NaivePriorityQueue()
er.insert(3, "Broken finger")
er.insert(1, "Heart attack")     # Most urgent!
er.insert(2, "Deep cut")
er.insert(5, "Common cold")
er.insert(1, "Stroke")           # Also most urgent!

print("ER Triage Order:")
while er.items:
    priority, patient = er.extract_min()
    print(f"  Priority {priority}: {patient}")

The naive approach **works** but is **slow** — sorting after every insert is O(n log n). We need something better: a **heap**.

---
## Part 2: The Heap Data Structure

A **heap** is a special kind of **complete binary tree** that satisfies the **heap property**:

- **Min-heap**: Every parent is **smaller than or equal to** its children (smallest at top)
- **Max-heap**: Every parent is **larger than or equal to** its children (largest at top)

Python's `heapq` module implements a **min-heap**.

### How a Heap is Stored in an Array

A heap is stored as a **flat list** (no tree nodes needed!):

```
         1           Index:  0
        / \                 / \
       3   2              1     2
      / \ / \           / \   / \
     7  4 5  6         3   4 5   6

List: [1, 3, 2, 7, 4, 5, 6]
```

| Relationship | Formula |
|-------------|--------|
| Parent of index `i` | `(i - 1) // 2` |
| Left child of index `i` | `2 * i + 1` |
| Right child of index `i` | `2 * i + 2` |

**Figure 2.1** — Verifying parent-child relationships in a heap array

In [ ]:
# A min-heap stored as a list
heap = [1, 3, 2, 7, 4, 5, 6]

print("Heap array:", heap)
print()

for i in range(len(heap)):
    parent_idx = (i - 1) // 2 if i > 0 else None
    left_idx = 2 * i + 1 if 2 * i + 1 < len(heap) else None
    right_idx = 2 * i + 2 if 2 * i + 2 < len(heap) else None

    parent_val = heap[parent_idx] if parent_idx is not None else "-"
    left_val = heap[left_idx] if left_idx is not None else "-"
    right_val = heap[right_idx] if right_idx is not None else "-"

    print(f"  Index {i} (value={heap[i]}): "
          f"parent={parent_val}, left={left_val}, right={right_val}")

**Figure 2.2** — Checking the min-heap property: every parent <= children

In [ ]:
def is_min_heap(lst):
    """Check if a list satisfies the min-heap property."""
    n = len(lst)
    for i in range(n):
        left = 2 * i + 1
        right = 2 * i + 2
        if left < n and lst[i] > lst[left]:
            return False
        if right < n and lst[i] > lst[right]:
            return False
    return True

print("[1, 3, 2, 7, 4, 5, 6] is min-heap?", is_min_heap([1, 3, 2, 7, 4, 5, 6]))
print("[1, 2, 3, 4, 5, 6, 7] is min-heap?", is_min_heap([1, 2, 3, 4, 5, 6, 7]))
print("[7, 3, 2, 1, 4, 5, 6] is min-heap?", is_min_heap([7, 3, 2, 1, 4, 5, 6]))

---
## Part 3: Python's `heapq` Module

Python provides the `heapq` module for heap operations. It works on regular lists.

| Function | Description | Time |
|----------|------------|------|
| `heapq.heappush(heap, item)` | Add item to heap | O(log n) |
| `heapq.heappop(heap)` | Remove & return smallest | O(log n) |
| `heapq.heapify(list)` | Convert list to heap in-place | O(n) |
| `heapq.nsmallest(k, iterable)` | Return k smallest items | O(n log k) |
| `heapq.nlargest(k, iterable)` | Return k largest items | O(n log k) |
| `heap[0]` | Peek at smallest (no removal) | O(1) |

**Figure 3.1** — Basic heapq operations: push and pop

In [ ]:
import heapq

# Start with an empty heap (just a list)
heap = []

# Push items
heapq.heappush(heap, 5)
heapq.heappush(heap, 2)
heapq.heappush(heap, 8)
heapq.heappush(heap, 1)
heapq.heappush(heap, 4)

print("Heap after pushes:", heap)
print("Smallest (peek):  ", heap[0])
print()

# Pop items (always returns the smallest)
print("Popping in order:")
while heap:
    print(f"  {heapq.heappop(heap)}", end="")
print()

**Figure 3.2** — Using heapify to convert an existing list into a heap

In [ ]:
import heapq

# Start with an unsorted list
data = [9, 4, 7, 1, 3, 8, 2]
print("Before heapify:", data)

# Convert to a heap IN-PLACE (O(n) time!)
heapq.heapify(data)
print("After heapify: ", data)
print("Smallest:      ", data[0])
print("Is min-heap?   ", is_min_heap(data))

**Figure 3.3** — ER triage with heapq (efficient version)

In [ ]:
import heapq

# ER patients: (priority, arrival_order, name)
# Using arrival_order as tiebreaker
er_heap = []
patients = [
    (3, "Broken finger"),
    (1, "Heart attack"),
    (2, "Deep cut"),
    (5, "Common cold"),
    (1, "Stroke"),
    (4, "Sprained ankle"),
]

for i, (priority, name) in enumerate(patients):
    heapq.heappush(er_heap, (priority, i, name))  # i breaks ties

print("ER Triage Order (heap-based):")
while er_heap:
    priority, _, patient = heapq.heappop(er_heap)
    print(f"  Priority {priority}: {patient}")

---
## Part 4: Finding the Smallest/Largest K Elements

A very common task: *"Give me the top 5 scores"* or *"Find the 3 cheapest products."*

### Approaches

| Approach | How | Time Complexity |
|----------|-----|----------------|
| Sort then slice | `sorted(data)[:k]` | O(n log n) |
| Heap-based | `heapq.nsmallest(k, data)` | O(n log k) |
| Full heap | heapify + pop k times | O(n + k log n) |

When `k << n`, the heap approach is significantly faster!

**Figure 4.1** — Using nsmallest and nlargest

In [ ]:
import heapq
import random

# Generate some exam scores
random.seed(42)
scores = [random.randint(30, 100) for _ in range(20)]
print("All scores:", scores)
print()

# Find top 5 and bottom 5
top5 = heapq.nlargest(5, scores)
bottom5 = heapq.nsmallest(5, scores)

print("Top 5 scores:   ", top5)
print("Bottom 5 scores:", bottom5)

**Figure 4.2** — Using nsmallest with a key function

In [ ]:
import heapq

# Find the 3 cheapest products
products = [
    {"name": "Laptop", "price": 999.99},
    {"name": "Mouse", "price": 29.99},
    {"name": "Keyboard", "price": 79.99},
    {"name": "Monitor", "price": 349.99},
    {"name": "USB Cable", "price": 9.99},
    {"name": "Headphones", "price": 149.99},
    {"name": "Webcam", "price": 59.99},
]

cheapest_3 = heapq.nsmallest(3, products, key=lambda p: p["price"])
print("3 Cheapest Products:")
for p in cheapest_3:
    print(f"  {p['name']:15s} ${p['price']:.2f}")

print()
most_expensive_3 = heapq.nlargest(3, products, key=lambda p: p["price"])
print("3 Most Expensive Products:")
for p in most_expensive_3:
    print(f"  {p['name']:15s} ${p['price']:.2f}")

**Figure 4.3** — Simulating a max-heap by negating values

In [ ]:
import heapq

# Python's heapq is a MIN-heap. To get a MAX-heap, negate the values!
max_heap = []

values = [5, 2, 8, 1, 9, 3]
for v in values:
    heapq.heappush(max_heap, -v)  # Negate on push

print("Popping in MAX order:")
while max_heap:
    val = -heapq.heappop(max_heap)  # Negate on pop to restore
    print(f"  {val}", end="")
print()

---
## Part 5: Benchmark — sorted() vs heapq.nsmallest()

> **🎯 Predict first, then measure. Does reality match your prediction?**
>
> Before running the benchmark below, write down your prediction: when `k` is much smaller than `n`, which method do you think will be faster — `sorted()[:k]` or `heapq.nsmallest(k)`? By how much?

Let's measure when `heapq.nsmallest()` beats `sorted()` and by how much.

**Figure 5.1** — Benchmark: finding k smallest from n elements

In [ ]:
import heapq
import time
import random

random.seed(42)
k = 10  # We want the 10 smallest
sizes = [1000, 5000, 10000, 50000, 100000, 500000, 1000000]

sorted_times = []
heap_times = []

for n in sizes:
    data = [random.random() for _ in range(n)]

    # Method 1: sorted() then slice
    t = 0
    for _ in range(3):
        start = time.perf_counter()
        result1 = sorted(data)[:k]
        t += time.perf_counter() - start
    sorted_times.append(t / 3)

    # Method 2: heapq.nsmallest
    t = 0
    for _ in range(3):
        start = time.perf_counter()
        result2 = heapq.nsmallest(k, data)
        t += time.perf_counter() - start
    heap_times.append(t / 3)

    speedup = sorted_times[-1] / heap_times[-1] if heap_times[-1] > 0 else float('inf')
    print(f"n={n:>10,}  sorted={sorted_times[-1]:.6f}s  "
          f"heapq={heap_times[-1]:.6f}s  speedup={speedup:.1f}x")

**Figure 5.2** — Plotting sorted() vs heapq.nsmallest() performance

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(sizes, sorted_times, 'ro-', label='sorted()[:k] — O(n log n)', linewidth=2)
plt.plot(sizes, heap_times, 'bs-', label='heapq.nsmallest(k) — O(n log k)', linewidth=2)
plt.xlabel('Number of Elements (n)', fontsize=12)
plt.ylabel('Time (seconds)', fontsize=12)
plt.title(f'Finding {k} Smallest Elements: sorted() vs heapq', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nWhen finding just {k} items from a large dataset,")
print("heapq.nsmallest() is faster because it doesn't sort everything.")
print("The advantage grows as n increases while k stays small.")

**Figure 5.3** — When does sorted() win? Testing with large k

In [ ]:
import heapq
import time
import random

random.seed(42)
n = 100_000
data = [random.random() for _ in range(n)]

k_values = [5, 10, 50, 100, 500, 1000, 5000, 10000]

print(f"n = {n:,}")
print(f"{'k':>8}  {'sorted()':>12}  {'heapq':>12}  {'Winner':>10}")
print("-" * 48)

for k in k_values:
    # sorted
    start = time.perf_counter()
    _ = sorted(data)[:k]
    t_sorted = time.perf_counter() - start

    # heapq
    start = time.perf_counter()
    _ = heapq.nsmallest(k, data)
    t_heap = time.perf_counter() - start

    winner = "heapq" if t_heap < t_sorted else "sorted"
    print(f"{k:>8}  {t_sorted:>10.6f}s  {t_heap:>10.6f}s  {winner:>10}")

print("\nRule of thumb: use heapq when k is much smaller than n.")
print("When k is close to n, sorted() can be faster.")

---
## Part 6: Common Errors & Pitfalls

**Figure 6.1** — Comparing uncomparable elements in a heap

In [ ]:
import heapq

# ERROR 1: Pushing tuples where the second element can't be compared
# This happens when priorities are equal and Python tries to compare the next element

class Task:
    def __init__(self, name):
        self.name = name

heap = []
heapq.heappush(heap, (1, Task("Write report")))

try:
    heapq.heappush(heap, (1, Task("Send email")))  # Same priority!
except TypeError as e:
    print(f"TypeError: {e}")
    print("Fix: Add a tiebreaker (like an index) between priority and the object")

# The fix:
heap = []
heapq.heappush(heap, (1, 0, Task("Write report")))  # index 0
heapq.heappush(heap, (1, 1, Task("Send email")))     # index 1
print(f"\nFixed! Heap has {len(heap)} items.")

**Figure 6.2** — Forgetting that heapq is a min-heap

In [ ]:
import heapq

# ERROR 2: Expecting max-heap behavior
scores = [85, 92, 78, 95, 88]
heapq.heapify(scores)

print("Popping from min-heap (lowest first):")
while scores:
    print(f"  {heapq.heappop(scores)}", end="")
print()

# If you wanted HIGHEST first, negate!
scores = [85, 92, 78, 95, 88]
max_scores = [-s for s in scores]
heapq.heapify(max_scores)

print("\nPopping from max-heap (highest first):")
while max_scores:
    print(f"  {-heapq.heappop(max_scores)}", end="")
print()

**Figure 6.3** — Indexing a heap (the list is NOT fully sorted!)

In [ ]:
import heapq

# ERROR 3: Thinking the heap list is sorted
data = [5, 3, 8, 1, 9, 2, 7]
heapq.heapify(data)

print("Heap list:  ", data)
print("Sorted list:", sorted(data))
print()
print("heap[0] IS the minimum:", data[0])
print("heap[1] is NOT necessarily the 2nd smallest!")
print(f"heap[1] = {data[1]}, but 2nd smallest = {sorted(data)[1]}")
print()
print("Only heap[0] is guaranteed. For the rest, you must pop.")

---
## 🌉 Bridge to Next Week

This week we learned about **heaps** and **priority queues** — structures that always give you the minimum (or maximum) element in O(log n) time.

The heap is actually a special case of a more general data structure: the **tree**. Notice how we described the heap as a "complete binary tree stored in an array."

**Next week**, we'll explore **trees** in depth:
- What is a tree? (root, nodes, leaves, depth)
- Building binary trees with linked nodes
- Three ways to walk through a tree: **in-order**, **pre-order**, and **post-order** traversal

See you in Week 9!

---
## 🎢 Exercises

Complete the exercises below. Make sure to **run each cell** after writing your solution.

### Easy Exercises

**EX1 (Easy):** Create a min-heap from the list `[15, 8, 23, 4, 42, 16, 3]` using `heapq.heapify()`. Print the heap and its minimum element.

Expected Output:
```
Heap: [3, 4, 15, 8, 42, 16, 23]
Minimum: 3
```
(Note: exact heap arrangement may vary, but minimum is always index 0)

<details><summary>💡 Hint</summary>
<code>heapq.heapify(data)</code> modifies the list in-place. The minimum is always at <code>data[0]</code>.
</details>

In [ ]:
# ✏️ [EX1] Your code here


**EX2 (Easy):** Push the numbers `10, 4, 15, 1, 7` one by one into an empty heap using `heapq.heappush()`. After each push, print the current state of the heap.

Expected Output:
```
Push 10: [10]
Push 4:  [4, 10]
Push 15: [4, 10, 15]
Push 1:  [1, 4, 15, 10]
Push 7:  [1, 4, 15, 10, 7]
```

<details><summary>💡 Hint</summary>
Start with <code>heap = []</code> and use a loop over the values. Print after each <code>heapq.heappush(heap, val)</code>.
</details>

In [ ]:
# ✏️ [EX2] Your code here


**EX3 (Easy):** Given a list of exam scores `[72, 85, 90, 65, 95, 88, 76, 92, 58, 83]`, use `heapq.nlargest()` to find the top 3 scores and `heapq.nsmallest()` to find the bottom 3 scores.

Expected Output:
```
Top 3:    [95, 92, 90]
Bottom 3: [58, 65, 72]
```

<details><summary>💡 Hint</summary>
Just call <code>heapq.nlargest(3, scores)</code> and <code>heapq.nsmallest(3, scores)</code>.
</details>

In [ ]:
# ✏️ [EX3] Your code here


**EX4 (Easy):** Pop all elements from a heap `[2, 5, 3, 8, 7, 6, 4]` (after heapifying) and collect them into a new list. Print the result. This is effectively a **heap sort**!

Expected Output:
```
Heap sorted: [2, 3, 4, 5, 6, 7, 8]
```

<details><summary>💡 Hint</summary>
First <code>heapq.heapify(data)</code>, then use a while loop: <code>result.append(heapq.heappop(data))</code> until the heap is empty.
</details>

In [ ]:
# ✏️ [EX4] Your code here


### Medium Exercises

**EX5 (Medium):** Implement a task scheduler using `heapq`. Create a list of tasks with priorities and names: `[(3, "Write report"), (1, "Fix critical bug"), (2, "Review PR"), (1, "Server down"), (4, "Update docs"), (2, "Code review")]`. Process them in priority order (lower number = higher priority). Use arrival order as tiebreaker.

Expected Output:
```
Processing tasks:
  [Priority 1] Fix critical bug
  [Priority 1] Server down
  [Priority 2] Review PR
  [Priority 2] Code review
  [Priority 3] Write report
  [Priority 4] Update docs
```

<details><summary>💡 Hint</summary>
Push tuples of <code>(priority, index, task_name)</code> to handle ties. The index ensures FIFO order for equal priorities.
</details>

In [ ]:
# ✏️ [EX5] Your code here


**EX6 (Medium):** Write a function `running_median(numbers)` that takes a list of numbers arriving one at a time and prints the median after each number arrives. Use two heaps: a max-heap for the lower half and a min-heap for the upper half.

Test with `[5, 15, 1, 3, 8]`.

Expected Output:
```
After 5: median = 5
After 15: median = 10.0
After 1: median = 5
After 3: median = 4.0
After 8: median = 5
```

<details><summary>💡 Hint</summary>
Use a max-heap (negate values) for the lower half and a min-heap for the upper half. Keep them balanced (sizes differ by at most 1). The median is the top of the larger heap, or the average of both tops if equal size.
</details>

In [ ]:
# ✏️ [EX6] Your code here


**EX7 (Medium):** Given a list of dictionaries representing students: `[{"name": "Alice", "gpa": 3.8}, {"name": "Bob", "gpa": 3.2}, {"name": "Charlie", "gpa": 3.9}, {"name": "Diana", "gpa": 3.5}, {"name": "Eve", "gpa": 3.7}, {"name": "Frank", "gpa": 3.1}]`, use `heapq.nlargest()` with a `key` parameter to find the top 3 students by GPA.

Expected Output:
```
Top 3 students by GPA:
  Charlie: 3.9
  Alice:   3.8
  Eve:     3.7
```

<details><summary>💡 Hint</summary>
Use <code>heapq.nlargest(3, students, key=lambda s: s["gpa"])</code>.
</details>

In [ ]:
# ✏️ [EX7] Your code here


**EX8 (Medium):** Write a function `merge_sorted_lists(lists)` that takes a list of sorted lists and merges them into one sorted list using `heapq.merge()`. Test with `[[1, 4, 7], [2, 5, 8], [3, 6, 9]]`.

Expected Output:
```
Merged: [1, 2, 3, 4, 5, 6, 7, 8, 9]
```

<details><summary>💡 Hint</summary>
Use <code>list(heapq.merge(*lists))</code> — the <code>*</code> unpacks the list of lists into separate arguments.
</details>

In [ ]:
# ✏️ [EX8] Your code here


**EX9 (Medium):** Write a function `kth_largest(nums, k)` that returns the k-th largest element in a list without sorting the entire list. Use a min-heap of size k.

Test: `kth_largest([3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5], 3)` should return `5`.

Expected Output:
```
3rd largest element: 5
```

<details><summary>💡 Hint</summary>
Maintain a min-heap of size k. For each number, if the heap has fewer than k elements, push it. Otherwise, if the number is larger than heap[0], pop the smallest and push the new number. At the end, heap[0] is the k-th largest.
</details>

In [ ]:
# ✏️ [EX9] Your code here


**EX10 (Medium):** Write a function `is_valid_min_heap(lst)` that checks whether a given list represents a valid min-heap. Test with both valid and invalid examples.

Expected Output:
```
[1, 3, 2, 7, 4, 5, 6] -> True
[1, 2, 3, 4, 5, 6, 7] -> True
[3, 1, 2, 7, 4, 5, 6] -> False
```

<details><summary>💡 Hint</summary>
For each index i, check that the element is less than or equal to its children at indices <code>2*i+1</code> and <code>2*i+2</code> (if they exist).
</details>

In [ ]:
# ✏️ [EX10] Your code here


### Challenge Exercises

**EX11 (Challenge):** Implement a `MinHeap` class from scratch (without using `heapq`). Implement `push(value)`, `pop()`, `peek()`, and helper methods `_sift_up()` and `_sift_down()`. Test by pushing `[5, 3, 8, 1, 2, 7]` and popping all elements.

<details><summary>💡 Hint</summary>
Store items in a list. On push, append and sift up (swap with parent while smaller). On pop, swap root with last element, remove last, then sift down (swap with smallest child while larger).
</details>

In [ ]:
# ✏️ [EX11] Your code here


**EX12 (Challenge):** Benchmark three approaches for finding the k smallest elements from n random numbers: (1) `sorted(data)[:k]`, (2) `heapq.nsmallest(k, data)`, (3) `heapq.heapify` + pop k times. Test with `n=500,000` and `k` values of `[5, 50, 500, 5000]`. Print a table and plot the results.

> **🎯 Predict first, then measure. Does reality match your prediction?** Before running, predict which method wins for each `k` value and why.

<details><summary>💡 Hint</summary>
For method 3, make a copy of the data first (<code>data_copy = data[:]</code>), then <code>heapq.heapify(data_copy)</code>, then pop k times in a loop. Time each method separately.
</details>

In [ ]:
# ✏️ [EX12] Your code here


---
## 📥 Submission

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📮 STEP 1: Fill in your info below, then run this cell
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STUDENT_ID    = ""     # e.g. "2024001234"
STUDENT_NAME  = ""     # e.g. "Ahmet Y\u0131lmaz"
STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"
CLASS_CODE    = ""     # code given in class
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import re as _re
_errors = []
if not _re.match(r"^\d{6,12}$", STUDENT_ID):
    _errors.append("\u274c Student ID must be 6-12 digits")
if len(STUDENT_NAME.strip().split()) < 2:
    _errors.append("\u274c Enter first and last name")
if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:
    _errors.append("\u274c Use your @istun.edu.tr email")
if len(CLASS_CODE.strip()) < 4:
    _errors.append("\u274c Invalid class code")
if _errors:
    for _e in _errors:
        print(_e)
    print("\n\u26a0\ufe0f  Fix the errors above and run this cell again.")
else:
    print(f"\u2705 Info OK \u2014 {STUDENT_NAME} ({STUDENT_ID})")
    print(f"   {STUDENT_EMAIL}")
    print(f"\n\ud83d\udc49 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📮 STEP 2: Run this cell to submit
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, re, os, urllib.request
WEEK = "Week_08"
URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"
try:
    _sid = STUDENT_ID.strip()
    _sname = STUDENT_NAME.strip()
    _semail = STUDENT_EMAIL.strip().lower()
    _scode = CLASS_CODE.strip().upper()
except NameError:
    raise SystemExit("\u274c Run the cell above first to set your info!")
if not _sid or not _sname or not _semail or not _scode:
    raise SystemExit("\u274c Run the cell above first \u2014 some fields are empty.")
_answers = {}
try:
    _ipy = get_ipython()
    _hist = _ipy.history_manager.get_range(output=False)
    for _sess, _line, _src in _hist:
        _m = re.match(r"#\s*\u270f\ufe0f\s*\[EX(\w+)\]", _src)
        if _m:
            _ex_id = "ex" + _m.group(1)
            _lines = _src.split("\n")
            _clean = "\n".join(_lines[1:]).strip()
            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}
except Exception:
    pass
if not _answers:
    try:
        for _src in In:
            if not _src: continue
            _m = re.match(r"#\s*\u270f\ufe0f\s*\[EX(\w+)\]", _src)
            if _m:
                _ex_id = "ex" + _m.group(1)
                _lines = _src.split("\n")
                _clean = "\n".join(_lines[1:]).strip()
                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}
    except NameError:
        pass
if not _answers:
    _nb_path = None
    try:
        _nb_path = __vsc_ipynb_file__
    except NameError:
        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]
        if len(_candidates) == 1: _nb_path = _candidates[0]
    if _nb_path and os.path.exists(str(_nb_path)):
        with open(str(_nb_path), "r", encoding="utf-8") as _f:
            _nb = json.load(_f)
        for _cell in _nb["cells"]:
            if _cell["cell_type"] != "code": continue
            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]
            _m = re.match(r"#\s*\u270f\ufe0f\s*\[EX(\w+)\]", _src)
            if _m:
                _ex_id = "ex" + _m.group(1)
                _lines = _src.split("\n")
                _clean = "\n".join(_lines[1:]).strip()
                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}
print(f"\ud83d\udcdd Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")
if not _answers:
    print("\n\u26a0\ufe0f  No exercise answers found!")
    print("Make sure you RAN all exercise cells before submitting.")
    raise SystemExit()
_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "dsa-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")
print("\ud83d\udce1 Submitting...")
try:
    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")
    _resp = urllib.request.urlopen(_req, timeout=30)
    _result = json.loads(_resp.read().decode())
    if _result.get("success"):
        print(f"\n\u2705 {_result['message']}")
        print("\ud83d\udce7 Check your email for confirmation.")
    else:
        print(f"\n\u274c {_result.get('message', 'Submission failed')}")
except Exception as _e:
    try:
        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")
        urllib.request.urlopen(_req, timeout=10)
    except: pass
    print(f"\n\u26a0\ufe0f  Request sent \u2014 check your email for confirmation.")
    print(f"(If no email arrives, try again or contact your instructor)")